In [1]:
!pip3 install pandas
import pandas as pd
import numpy as np
import json

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


## Define the metadata

In [42]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41590-021-00922-4"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41590_2021_922_MOESM2_ESM.xlsx"
table_name = "paper_degs.xlsx" # local name # UPDATE THIS TO BE SPECIFIC FOR HILDRETH

## Read in file

In [43]:
excel = pd.read_excel(table_name, sheet_name= "Cluster_Markers_CD45PosNeg", skiprows = 2)

df = excel

In [44]:
df.head()

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
0,0.000000e+00,0.911908,0.269,0.014,0.000000e+00,CD8 gdT,LINC02446
1,0.000000e+00,0.905986,0.464,0.043,0.000000e+00,CD8 gdT,LEF1
2,0.000000e+00,0.748453,0.301,0.016,0.000000e+00,CD8 gdT,FXYD2
3,0.000000e+00,0.727598,0.262,0.006,0.000000e+00,CD8 gdT,ZNF683
4,4.540614e-241,0.962575,0.329,0.031,1.242811e-236,CD8 gdT,KLRC2


## Perform the filtering

In [45]:
col_pval = "p_val_adj"
col_lfc = "avg_logFC"
col_gene = "gene"
col_ct = "cluster" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [46]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [47]:
markers

,p_val,avg_logFC,pct.1,pct.2,p_val_adj,cluster,gene
25593,0.0,2.495286,0.166,0.003,0.0,B cell,IGHG3
25591,0.0,3.038060,0.173,0.003,0.0,B cell,IGHG1
25592,0.0,2.512809,0.193,0.001,0.0,B cell,IGLC3
25588,0.0,3.649848,0.289,0.012,0.0,B cell,IGHA1
25587,0.0,3.768381,0.295,0.004,0.0,B cell,IGLC2
...,...,...,...,...,...,...,...
17859,0.0,2.512966,0.999,0.261,0.0,APC,MFAP5
20178,0.0,2.411521,1.000,0.213,0.0,Preadipocyte,CFD
20177,0.0,2.502895,1.000,0.232,0.0,Preadipocyte,GSN
14585,0.0,2.210930,1.000,0.721,0.0,ncMo,SAT1


In [48]:
markers[col_ct].value_counts()

cluster
Endothelial      15
Smooth muscle    11
Myeloid-like     10
PVM              10
B cell            9
Preadipocyte      9
ncMo              6
APC               4
mNK               3
NK-like           2
cDC2B             2
cDC1              1
Name: count, dtype: int64

In [35]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster
B cell           0.395778
Myeloid-like     0.765900
Endothelial      0.793067
Smooth muscle    0.850091
NK-like          0.859000
APC              0.907250
PVM              0.924200
ncMo             0.961833
Preadipocyte     0.966222
cDC2B            0.971500
mNK              0.977000
cDC1             0.996000
Name: pct.1, dtype: float64

## Convert to evidence json

In [36]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "gene"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["gene_id"] = None
save = df.to_dict(orient = "records")

In [37]:
save

[{'cell_type_label': 'B cell',
  'gene': 'IGHG3',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'IGHG1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'IGLC3',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'IGHA1',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'IGLC2',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'B cell',
  'gene': 'JCHAIN',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'Myeloid-like',
  'gene': 'S100A8',
  'organism': 'homo_sapiens',
  'cell_source': 'adipose',
  'cell_state': None},
 {'cell_type_label': 'Myeloid-like',
  'gene': 'CCL20',
  'organism': 'homo

## Save evidence

In [38]:
with open("evidence.json", "w") as f:
    json.dump(save, f, indent = 4) 